In [1]:
import os
import json
from tqdm import tqdm
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import HDBSCAN, OPTICS
from sklearn.preprocessing import normalize
import plotly.express as px
from sklearn.manifold import TSNE

from extraction_prompts import (
    FIND_BETTER_ENTITY_NAME,
    VALIDATE_ENTITY_NAME,
    DECIDE_OUTLIER_FATE_ENTITY,
    DECIDE_OUTLIER_FATE_RELATION,
    FIND_BETTER_RELATION_NAME,
    VALIDATE_RELATION_NAME
)

from utils import ollama_request, EMBEDD_MODEL_1

INDEX_NAME = os.getenv("INDEX_NAME")

/home/zbrzeznyg/miniconda3/envs/masters/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
triplets = pd.read_csv("data/triplets.csv")

In [3]:
def create_data_mapping(data, embed_model, cluster_model, outlier_prompt, replace_prompt, validate_prompt, N=2, perform_validation=True):
    mapping = {}
    valid_data = []
    for term in tqdm(data, desc="Validating terms..."):
        is_valid = ollama_request(
            prompt=outlier_prompt.format(outlier=term),
            is_stream=False
        ).replace('"', "").replace("'", "").replace("*", "").strip().lower()
        # print(f"Term {term} validation result: {is_valid}")
        if is_valid.strip().lower().replace('"', "").replace("'", "") == "valid":
            valid_data.append(term)
        else:
            mapping[str(term)] = None
    if not valid_data:
        return mapping
    encoded_entities = embed_model.encode(valid_data, normalize_embeddings=True).tolist()
    X = normalize(encoded_entities, norm="l2")
    try:
        cluster_labels = cluster_model.fit_predict(X)
    except Exception as e:
        outliers = [-1 for _ in range(len(valid_data))]
    outliers = np.array(valid_data)[np.where(cluster_labels == -1)[0]]
    for outlier in outliers:
        mapping[str(outlier)] = str(outlier)


    for i in tqdm(range(max(cluster_labels) + 1), desc="Finding better names..."):
        cluster_indices = np.where(cluster_labels == i)[0]
        cluster_entities = np.array(valid_data)[cluster_indices]
        encoded_cluster_entities = np.array(encoded_entities)[cluster_indices]
        cluster_mean = np.mean(encoded_cluster_entities, axis=0)
        distances = np.linalg.norm(encoded_cluster_entities - cluster_mean, axis=1)
        closest_entities = cluster_entities[distances <= min(distances)+(max(distances) - min(distances))/N]
        for attempt in range(2):
            better_entity_name = ollama_request(
                prompt=replace_prompt.format(list=closest_entities.tolist()),
                is_stream=False
            ).strip().lower().replace('"', "").replace("'", "").replace("*", "")

            # print(f"Processing closest entities: {closest_entities.tolist()}")
            # print(f"Better entity name suggestion: {better_entity_name}")

            if better_entity_name == "do_not_merge":
                for entity in cluster_entities:
                    mapping[str(entity)] = str(entity)
                break

            if perform_validation:
                is_valid_entity_name = ollama_request(
                    prompt=validate_prompt.format(name=better_entity_name),
                    is_stream=False
                )
            else:
                is_valid_entity_name = "valid"

            if is_valid_entity_name.strip().lower().replace('"', "").replace("'", "") == "valid":
                for entity in cluster_entities:
                    mapping[str(entity)] = better_entity_name
                break
            else:
                if attempt == 1:
                    for entity in cluster_entities:
                        # print(f"Could not find a valid better name for entity: {entity}. Keeping the original name.")
                        mapping[str(entity)] = str(entity)
    return mapping, valid_data, encoded_entities, cluster_labels

In [4]:
def viz_clusters(valid_data, encoded_entities, cluster_labels):
    embeddings = np.array(encoded_entities)
    cluster_labels = np.array(cluster_labels)

    embeddings_2d = TSNE(
        n_components=2,
        perplexity=30,
        random_state=42
    ).fit_transform(embeddings)

    plot_df = pd.DataFrame({
        "x": embeddings_2d[:, 0],
        "y": embeddings_2d[:, 1],
        "name": valid_data,
        "cluster": cluster_labels
    })

    plot_df["cluster"] = plot_df["cluster"].astype(str)

    fig = px.scatter(
        plot_df,
        x="x",
        y="y",
        color="cluster",
        hover_name="name",
        hover_data={
            "cluster": True,
            "x": False,
            "y": False
        },
        title="Entity embedding clusters"
    )

    fig.update_traces(
        marker=dict(size=8)
    )

    fig.update_layout(
        xaxis_title="t-SNE 1",
        yaxis_title="t-SNE 2",
        legend_title="Cluster"
    )

    fig.show()

### Entities

In [5]:
unique_entities = list(set(pd.concat([triplets["subject"], triplets["object"]]).dropna()))
embed_model_entities = EMBEDD_MODEL_1

cluster_model_entities = OPTICS(
    min_samples=2,
    min_cluster_size=2,
    metric="cosine",
    cluster_method="xi",
    xi=0.03,
)

outlier_prompt_entity = DECIDE_OUTLIER_FATE_ENTITY
replace_prompt_entity = FIND_BETTER_ENTITY_NAME
validate_prompt_entity = VALIDATE_ENTITY_NAME
N_entities = 2

entities_mapping, valid_data_entities, encoded_entities, cluster_labels_entities = create_data_mapping(
    data=unique_entities,
    embed_model=embed_model_entities,
    cluster_model=cluster_model_entities,
    outlier_prompt=outlier_prompt_entity,
    replace_prompt=replace_prompt_entity,
    validate_prompt=validate_prompt_entity,
    N=N_entities,
    perform_validation=False
)

viz_clusters(valid_data_entities, encoded_entities, cluster_labels_entities)

with open("data/entities_mapping.json", "w") as f:
    json.dump(entities_mapping, f, indent=4)

Validating terms...: 100%|██████████| 12494/12494 [57:37<00:00,  3.61it/s] 
/home/zbrzeznyg/miniconda3/envs/masters/lib/python3.11/site-packages/sklearn/cluster/_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
Finding better names...: 100%|██████████| 2205/2205 [13:20<00:00,  2.75it/s]


### Predicates

In [6]:
unique_predicates = list(set(triplets["predicate"].dropna()))
embed_model_predicates = EMBEDD_MODEL_1

cluster_model_predicates = HDBSCAN(
    min_cluster_size=2,
    min_samples=2,
    metric="cosine",
    cluster_selection_method="leaf",
    cluster_selection_epsilon=0.0,
)

outlier_prompt_predicate = DECIDE_OUTLIER_FATE_RELATION
replace_prompt_predicate = FIND_BETTER_RELATION_NAME
validate_prompt_predicate = VALIDATE_RELATION_NAME
N_predicates = 2

predicate_mapping, valid_data_predicates, encoded_predicates, cluster_labels_predicates = create_data_mapping(
    data=unique_predicates,
    embed_model=embed_model_predicates,
    cluster_model=cluster_model_predicates,
    outlier_prompt=outlier_prompt_predicate,
    replace_prompt=replace_prompt_predicate,
    validate_prompt=validate_prompt_predicate,
    N=N_predicates,
    perform_validation=False
)

viz_clusters(valid_data_predicates, encoded_predicates, cluster_labels_predicates)

with open("data/predicate_mapping.json", "w") as f:
    json.dump(predicate_mapping, f, indent=4)

Finding better names...: 100%|██████████| 2111/2111 [12:41<00:00,  2.77it/s]


### Trimm triplets

In [7]:
entities_mapping = json.load(open("data/entities_mapping.json", "r"))
predicate_mapping = json.load(open("data/predicate_mapping.json", "r"))

new_triplets = pd.DataFrame(columns=triplets.columns)
for t in tqdm(range(triplets.shape[0]), "Trimming entities..."):
    triplet = triplets.iloc[t].copy()
    new_subject = entities_mapping.get(str(triplet["subject"]), None)
    new_object = entities_mapping.get(str(triplet["object"]), None)
    new_predicate = predicate_mapping.get(str(triplet["predicate"]), None)
    if not new_subject or not new_object or not new_predicate:
        continue
    else:
        triplet["subject"] = new_subject
        triplet["object"] = new_object
        triplet["predicate"] = new_predicate

    new_triplets = pd.concat([new_triplets, pd.DataFrame([triplet])], ignore_index=True)

new_triplets = new_triplets.drop_duplicates()
new_triplets.to_csv("data/new_triplets.csv", index=False)

Trimming entities...: 100%|██████████| 18093/18093 [00:05<00:00, 3146.76it/s]
